In [ ]:
import pandas as pd
import numpy as np
import pickle

from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split




df = pd.read_csv("DiseaseAndSymptoms.csv")
df.fillna("None", inplace=True)




symptoms = set()

for col in df.columns[1:]:
    symptoms.update(df[col].values)

symptoms.discard("None")
symptoms = sorted(symptoms)




X = []

for index, row in df.iterrows():
    row_symptoms = row[1:].values
    temp = [1 if symptom in row_symptoms else 0 for symptom in symptoms]
    X.append(temp)

X = np.array(X)




le = LabelEncoder()
y = le.fit_transform(df["Disease"])




X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = RandomForestClassifier()
model.fit(X_train, y_train)

print("Accuracy:", model.score(X_test, y_test))




with open("disease_model.pkl", "wb") as f:
    pickle.dump(model, f)

with open("label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)

with open("symptoms.pkl", "wb") as f:
    pickle.dump(symptoms, f)

print("Model Saved Successfully ✅")




with open("disease_model.pkl", "rb") as f:
    model = pickle.load(f)

with open("label_encoder.pkl", "rb") as f:
    le = pickle.load(f)

with open("symptoms.pkl", "rb") as f:
    symptoms = pickle.load(f)




def predict_disease(input_symptoms):
    input_vector = [1 if symptom in input_symptoms else 0 for symptom in symptoms]
    input_vector = np.array(input_vector).reshape(1, -1)

    pred = model.predict(input_vector)
    return le.inverse_transform(pred)[0]




result = predict_disease(['fever', 'cough', 'headache'])
print("Predicted Disease:", result)